# Figure 1 — Method Pipeline

Reproduces **Figure 1** of the paper (see also `pipeline.png` in this repo/`README`) -- the four-panel overview of the method (distance distributions in embedding space and Fisher space, t-SNE projection, label-propagation classification of an unseen response).

---

## Setup

In [ ]:
import os
import pathlib

# allow execution from either the repo root or code/
ROOT = pathlib.Path.cwd()
if ROOT.name == "code":
    os.chdir(ROOT.parent)


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import aporia as ap

os.chdir(pathlib.Path.cwd()/"code")
from _pipeline_helpers import *
os.chdir(pathlib.Path.cwd()/"..")


In [ ]:
# ---------------------------------------------------------------
# Load dataset + model metadata from a TOML config.
# ---------------------------------------------------------------
CONFIG_PATH = "config/socrates.toml"

cfg = ap.load_config(CONFIG_PATH)

model_names           = cfg.model_names
figures_dir           = cfg.cache.fig_dir
best_reg_lambda       = cfg.experiment.best_lambda
maxResponsesPerPrompt = cfg.dataset.max_responses_per_prompt


In [ ]:
plt.rcParams['text.usetex'] = True
plt.rcParams['text.latex.preamble'] = ap.matplotlib_latex_preamble(cfg)


## Data

In [ ]:
df = ap.load_dataframe(cfg)


In [ ]:
model_order, model_rank = ap.build_model_size_order(cfg)

false_premise = (
    {pid: fp for pid, fp in df[["prompt_id", "false_premise"]].value_counts().keys()}
    if "false_premise" in df.columns
    else None
)


## Experiment

In [ ]:
results_df, geometry_store, null_store = ap.run_structural_analysis(
    df, cfg,
    use_cache=True,
    overwrite_cache=False,
)


## Plots

In [ ]:
mid = 2  # model id
pid = 82 # prompt ic
rid = 18 # response id (it was 11 on the old dataset)

## Figures and tables

In [ ]:
plot_distance_violin(
    geometry_store,
    key=(mid, pid),
    space="embedding",
    savepath=f"{figures_dir}/P_violin_embedding.pdf",
    transparent=True,
    rotate=True
)
plot_distance_violin(
    geometry_store,
    key=(mid, pid),
    space="fisher",
    savepath=f"{figures_dir}/P_violin_fisher.pdf",
    transparent=True,
    rotate=True
)

In [ ]:


X, y = ap.extract_prompt_data(df, 2, 82, cfg)

X_trn, X_tst, y_trn, y_tst, trainIdxs, testIdxs = train_test_split_80_20(
    X, y, random_state=42
)

# ---- t-SNE ----
Z_tr, Z_te, y_tr, y_te = tsne_projection(
    X_trn, y_trn,
    X_tst, y_tst,
    perplexity=30,
    random_state=42,
)

# ---- graph ----
G = build_tsne_complete_graph(Z_tr, y_tr)

# ---- plot ----
fig, ax = plot_tsne_complete_graph(G, edge_lw=None)
fig.savefig(f'{figures_dir}/P_embedding_edgeless.pdf', bbox_inches='tight', transparent=True)
fig, ax = plot_tsne_complete_graph(G, )
fig.savefig(f'{figures_dir}/P_embedding.pdf', bbox_inches='tight', transparent=True)

### Test point, ie LP

In [ ]:
X, y = ap.extract_prompt_data(df, 2, 82, cfg)

X_trn, X_tst, y_trn, y_tst, _, _ = train_test_split_80_20(
    X, y, random_state=42
)

Z_tr, Z_te, y_tr, y_te = tsne_projection(
    X_trn, y_trn,
    X_tst, y_tst,
    perplexity=30,
    random_state=42,
)
G_star = build_tsne_star_graph(
    Z_tr,
    y_tr,
    Z_te[rid],
)

fig, ax = plot_tsne_star_graph(G_star, edge_lw=None)
fig.savefig(f"{figures_dir}/P_embedding_LP{rid}_edgeless.pdf", bbox_inches="tight", transparent=True)

fig, ax = plot_tsne_star_graph(G_star)
fig.savefig(f"{figures_dir}/P_embedding_LP{rid}.pdf", bbox_inches="tight", transparent=True)

In [ ]:
# ---- fit Fisher ----
X_G, X_H = ap.split_by_label(X_trn, y_trn)
v = ap.fisher_direction(X_G, X_H, lambda_reg=best_reg_lambda)

z_tr = X_trn @ v
z_te = X_tst @ v

Z_tr = fisher_to_oblique(z_tr, angle_deg=-30)
Z_te = fisher_to_oblique(z_te, angle_deg=-30)
Z_tr_j = add_orthogonal_jitter(Z_tr, scale=0.01)

# ---- training scatter + optional test ----
fig, ax = plot_fisher_with_optional_test(
    Z_tr_j, y_tr,
    z_test=None,
    angle_deg=-30,
)

# ---- Fisher axis ----
draw_fisher_axis(ax, z_tr, angle_deg=-30, lw=1.5)

# ---- optional violins (unchanged) ----
# draw_fisher_violin(...)
# draw_fisher_violin(...)

ax.invert_yaxis()
fig.savefig(f"{figures_dir}/P_fisher.pdf", bbox_inches="tight", transparent=True)




# ---- training scatter + optional test ----
fig, ax = plot_fisher_with_optional_test(
    Z_tr_j, y_tr,
    z_test=z_te[rid],
    angle_deg=-30,
)

# ---- Fisher axis ----
draw_fisher_axis(ax, z_tr, angle_deg=-30, lw=1.5)

# ---- optional violins (unchanged) ----
# draw_fisher_violin(
#     ax,
#     z_tr[y_tr == 0],
#     angle_deg=-30,
#     width=0.07,
#     color=COLORS[0],
#     alpha=0.35,
#     zorder=1,
#     dobot=False
# )

# draw_fisher_violin(
#     ax,
#     z_tr[y_tr == 1],
#     angle_deg=-30,
#     width=0.07,
#     color=COLORS[1],
#     alpha=0.35,
#     zorder=1,
#     dotop=False
# )

ax.invert_yaxis()
fig.savefig(f"{figures_dir}/P_fisher_LP{rid}.pdf", bbox_inches="tight", transparent=True)

In [ ]:
dist_df, wass_df = generate_fisher_distance_table(
    df,
    mid,
    pid,
    cfg,
    lambda_reg=best_reg_lambda,
    random_state=42
)

tid2H = {}
for k, v in dist_df.groupby(['test_id', 'y_true']):
    print(int(k[0]), k[1]==1)
    tid2H[int(k[0])] = k[1]==1


# Fisher space
plot_test_distance_violin(
    dist_df,
    test_id=rid,
    space="fisher",
    savepath=f"{figures_dir}/P_violin_fisher_LP{rid}.pdf",
    transparent = True,
    rotate=True
)

# Original embedding space
plot_test_distance_violin(
    dist_df,
    test_id=rid,
    space="embedding",
    savepath=f"{figures_dir}/P_violin_embedding_LP{rid}.pdf",
    transparent = True,
    rotate=True
)